# Exercice 1 — Nettoyage et encodage

In [15]:
import numpy as np

import pandas as pd


from sklearn.impute import SimpleImputer 

from sklearn.preprocessing import OrdinalEncoder , OneHotEncoder

### Creation du dataframe avec max de correlation

In [16]:
rng = np.random.default_rng(42)

n = 200

age = rng.normal(loc=35, scale=12, size=n)

age = np.clip(age, 18, 80).astype(int)

niveau_education = []

for a in age:

    if a < 25:
        choix = ["Collège", "Lycée", "Bac+2", "Bac+3", "Bac+5", "Doctorat"]

        proba = [0.05, 0.30, 0.30, 0.20, 0.12, 0.03]

    elif a < 35:
        choix = ["Lycée", "Bac+2", "Bac+3", "Bac+5", "Doctorat"]

        proba = [0.15, 0.25, 0.25, 0.28, 0.07]

    else:
        choix = ["Lycée", "Bac+2", "Bac+3", "Bac+5", "Doctorat"]

        proba = [0.20, 0.25, 0.20, 0.25, 0.10]

    niveau_education.append(

        rng.choice(choix, p=proba)

    )

ville = rng.choice(
    ["Paris", "Lyon", "Marseille", "Toulouse", "Nice", "Nantes"],
    size=n,
    p=[0.30, 0.15, 0.15, 0.15, 0.10, 0.15]
)

genre = rng.choice(
    ["Homme", "Femme", "Autre"],
    size=n,
    p=[0.48, 0.48, 0.04]
)

statut_marital = []

for a in age:

    if a < 25:
        choix = ["Célibataire", "Marié", "Divorcé", "Veuf"]
        proba = [0.85, 0.12, 0.03, 0.00]

    elif a < 35:
        choix = ["Célibataire", "Marié", "Divorcé", "Veuf"]
        proba = [0.45, 0.45, 0.09, 0.01]

    elif a < 50:
        choix = ["Célibataire", "Marié", "Divorcé", "Veuf"]
        proba = [0.25, 0.60, 0.13, 0.02]

    else:
        choix = ["Célibataire", "Marié", "Divorcé", "Veuf"]
        proba = [0.15, 0.60, 0.15, 0.10]

    statut_marital.append(
        rng.choice(choix, p=proba)
    )

taille_cm = rng.normal(
    loc=172,
    scale=9,
    size=n
)

taille_cm = np.clip(taille_cm, 150, 200).round(1)

effet_education = {
    "Collège": 0,
    "Lycée": 5000,
    "Bac+2": 10000,
    "Bac+3": 15000,
    "Bac+5": 25000,
    "Doctorat": 35000
}

effet_ville = {
    "Paris": 18000,
    "Lyon": 9000,
    "Marseille": 4000,
    "Toulouse": 6000,
    "Nice": 7000,
    "Nantes": 5000
}

salaire_annuel = (
    18000
    + age * 700
    + np.array([effet_education[x] for x in niveau_education])
    + np.array([effet_ville[x] for x in ville])
    + rng.normal(0, 10000, n)
)

salaire_annuel = np.clip(
    salaire_annuel,
    20000,
    150000
).round(0)

df = pd.DataFrame({
    "age": age,
    "salaire_annuel": salaire_annuel,
    "taille_cm": taille_cm,
    "niveau_education": niveau_education,
    "ville": ville,
    "genre": genre,
    "statut_marital": statut_marital
})


In [17]:
df

,age,salaire_annuel,taille_cm,niveau_education,ville,genre,statut_marital
0,38,35519.0,171.0,Lycée,Nantes,Autre,Célibataire
1,22,42231.0,174.0,Lycée,Nantes,Femme,Célibataire
2,44,61821.0,177.6,Lycée,Paris,Femme,Marié
3,46,81351.0,163.0,Bac+5,Nice,Homme,Célibataire
4,18,37598.0,162.6,Lycée,Paris,Femme,Célibataire
...,...,...,...,...,...,...,...
195,24,39530.0,194.1,Bac+2,Toulouse,Femme,Célibataire
196,42,74686.0,188.2,Bac+2,Marseille,Femme,Célibataire
197,30,83456.0,168.3,Bac+5,Nice,Homme,Marié
198,34,81396.0,168.7,Bac+5,Nantes,Femme,Célibataire


### Injection des valeurs manquantes

In [18]:
operation_injection = {

    "age" : 15,

    "salaire_annuel" : 20,

    "taille_cm" : 10,

    "niveau_education" : 12,

    "ville" : 8,

    "genre" : 5,

    "statut_marital" : 10


}


for  var , n  in operation_injection.items() :

    indice = np.random.choice(

        df.index,

        size = n,

        replace=False,

    )

    df.loc[indice, var] = np.nan

In [19]:
df.to_csv("dataset/ex1_dataset_non_traité.csv",index=False)

df.isnull().sum()

age                 15
salaire_annuel      20
taille_cm           10
niveau_education    12
ville                8
genre                5
statut_marital      10
dtype: int64

### Imputation des 3 colonnes numériques par la médiane

In [20]:
num_col = ["age","salaire_annuel","taille_cm"]

operation = SimpleImputer(strategy="mean")

df[num_col] = operation.fit_transform(df[num_col])


### Imputation des 4 colonnes catégorielles par la valeur la plus fréquente

In [21]:
cat_col = ["niveau_education","ville","genre","statut_marital"]

operation = SimpleImputer(strategy="most_frequent")

df[cat_col] = operation.fit_transform(df[cat_col])


In [22]:
df.to_csv("dataset/ex1_dataset_traité.csv",index=False)


df.isnull().sum()

age                 0
salaire_annuel      0
taille_cm           0
niveau_education    0
ville               0
genre               0
statut_marital      0
dtype: int64

### Encode niveau_education avec un OrdinalEncoder en respectant explicitement l'ordre ci-dessus

In [23]:
encoder  =  OrdinalEncoder()

encoder.fit(df[["niveau_education"]])

df["niveau_education"] = encoder.transform(df[["niveau_education"]])


In [24]:
df

,age,salaire_annuel,taille_cm,niveau_education,ville,genre,statut_marital
0,38.0,35519.0,171.0,4.0,Nantes,Autre,Célibataire
1,22.0,42231.0,174.0,4.0,Nantes,Femme,Célibataire
2,44.0,61821.0,177.6,4.0,Paris,Femme,Marié
3,46.0,81351.0,163.0,2.0,Nice,Homme,Célibataire
4,18.0,37598.0,162.6,4.0,Paris,Femme,Célibataire
...,...,...,...,...,...,...,...
195,24.0,39530.0,194.1,0.0,Toulouse,Femme,Célibataire
196,42.0,74686.0,188.2,0.0,Marseille,Homme,Célibataire
197,30.0,83456.0,168.3,2.0,Nice,Homme,Marié
198,34.0,81396.0,168.7,2.0,Nantes,Femme,Célibataire


### Encodage de ville, genre et statut_marital avec un encodage one-hot (pd.get_dummies ou OneHotEncoder

In [25]:
col = ["ville","genre","statut_marital"]

encoder_vgs = OneHotEncoder(sparse_output=False)

tab = encoder_vgs.fit_transform(df[col])

df_encode = pd.DataFrame(
    tab,
    columns=encoder_vgs.get_feature_names_out(col),
    index=df.index
)

df = df.drop(col,axis=1)

df = df.join(df_encode)

df

,age,salaire_annuel,taille_cm,niveau_education,ville_Lyon,ville_Marseille,ville_Nantes,ville_Nice,ville_Paris,ville_Toulouse,genre_Autre,genre_Femme,genre_Homme,statut_marital_Célibataire,statut_marital_Divorcé,statut_marital_Marié,statut_marital_Veuf
0,38.0,35519.0,171.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1,22.0,42231.0,174.0,4.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
2,44.0,61821.0,177.6,4.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,46.0,81351.0,163.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
4,18.0,37598.0,162.6,4.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,24.0,39530.0,194.1,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
196,42.0,74686.0,188.2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0
197,30.0,83456.0,168.3,2.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
198,34.0,81396.0,168.7,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


### Vérifions avec .isna().sum() qu'il ne reste aucune valeur manquante, et avec .dtypes qu'il ne reste aucune colonne non numérique.

In [26]:
df.isna().sum()

age                           0
salaire_annuel                0
taille_cm                     0
niveau_education              0
ville_Lyon                    0
ville_Marseille               0
ville_Nantes                  0
ville_Nice                    0
ville_Paris                   0
ville_Toulouse                0
genre_Autre                   0
genre_Femme                   0
genre_Homme                   0
statut_marital_Célibataire    0
statut_marital_Divorcé        0
statut_marital_Marié          0
statut_marital_Veuf           0
dtype: int64